# 🏆 Predicción del Mundial 2026 — ML + econometría

Notebook **auto-actualizable**: lee el Excel `Mundial_2026_fuente_datos.xlsx`,
toma los resultados ya cargados como hechos fijos y **recalcula** las
probabilidades cada vez que se reejecuta (`Entorno de ejecución ▸ Ejecutar todo`).

**Modelos:** Elo probabilístico (baseline) + Dixon-Coles/Poisson regularizado
con prior Elo (econometría) + logit multinomial, RandomForest y GradientBoosting
(ML, complementarios por la muestra chica), combinados en un **ensemble**, y un
**Monte Carlo** del torneo completo (grupos → final) con el desempate oficial
FIFA y la asignación de los 8 mejores terceros.

> ⚠️ Este trabajo entrega **probabilidades**, no consejos de apuestas.


## 1. Preparación del entorno (clona el repo e instala dependencias en Colab)

In [ ]:
import os, sys

EN_COLAB = 'google.colab' in sys.modules
REPO = 'ML_prediccion_mundial2026'
REPO_URL = 'https://github.com/santiagoriverti/ML_prediccion_mundial2026.git'

# En Colab: clonar el repo (trae src/ y el Excel) e instalar requirements.
if EN_COLAB:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL}
    %cd {REPO}
    !pip install -q -r requirements.txt

# Localizar la carpeta src/ (funciona tanto en Colab como en local).
for cand in ['src', '../src', os.path.join(REPO, 'src')]:
    if os.path.exists(cand):
        sys.path.insert(0, os.path.abspath(cand))
        RAIZ = os.path.dirname(os.path.abspath(cand))
        break
print('Raíz del proyecto:', RAIZ)


## 2. Carga del Excel (siempre la última versión commiteada)

In [ ]:
import glob, urllib.request
import warnings; warnings.filterwarnings('ignore')

# 1º intenta la raw URL del repo (toma el último commit); 2º archivo local;
# 3º (sólo Colab) subida manual.
RAW = ('https://raw.githubusercontent.com/santiagoriverti/'
       'ML_prediccion_mundial2026/main/Mundial_2026_fuente_datos.xlsx')
datos_bytes = None
try:
    datos_bytes = urllib.request.urlopen(RAW, timeout=25).read()
    print('✓ Excel cargado desde la raw URL del repo.')
except Exception as e:
    print('La raw URL falló (', e, '); busco archivo local .xlsx...')
    locales = glob.glob(os.path.join(RAIZ, '*.xlsx')) + glob.glob('*.xlsx')
    if locales:
        datos_bytes = open(locales[0], 'rb').read()
        print('✓ Excel local:', locales[0])
    elif EN_COLAB:
        from google.colab import files
        subido = files.upload()
        datos_bytes = list(subido.values())[0]
        print('✓ Excel subido manualmente.')
    else:
        raise FileNotFoundError('No se encontró el Excel insumo.')


## 3. Lectura y limpieza de datos

In [ ]:
from data_loader import cargar_datos
from features import imputar_rating_base, construir_dataset_partidos

datos = cargar_datos(datos_bytes)
print('Equipos:', datos.meta['n_equipos'],
      '| Partidos de grupo:', datos.meta['n_partidos_grupo'],
      '| Ya jugados:', datos.meta['n_jugados'],
      '| Sin puntos FIFA (imputados):', datos.meta['n_sin_puntos'])

# Rating base de fuerza (derivado de Puntos FIFA con imputación de faltantes).
equipos = imputar_rating_base(datos.equipos)
equipos[['pais', 'grupo', 'confederacion', 'rating_base', 'rating_imputado']] \
    .sort_values('rating_base', ascending=False).head(10)


## 4. Actualización del Elo con los resultados ya cargados

Antes de simular, el rating de cada selección se mueve con los partidos
**ya jugados** (los hechos fijan el pronóstico).

In [ ]:
from simulate import actualizar_elo

equipos = actualizar_elo(equipos, datos.fixture, K=32.0)
dataset = construir_dataset_partidos(equipos, datos.fixture)
print('Dataset por partido:', dataset.shape, '| con resultado (jugados):',
      int(dataset['jugado'].sum()))
equipos[['pais', 'rating_base']].sort_values('rating_base', ascending=False).head(8)


## 5. Entrenamiento de los modelos (Dixon-Coles + ML)

In [ ]:
from models import DixonColes, entrenar_modelos_ml

# Econometría: Dixon-Coles con prior basado en el rating Elo (regularización
# fuerte por la muestra chica).
dc = DixonColes(equipos).entrenar(dataset)

# ML supervisado (complementario): logit multinomial, RandomForest, GBM, con
# validación cruzada y calibración.
modelos_ml, reporte_ml = entrenar_modelos_ml(dataset)
print('ML — n partidos de entrenamiento:', reporte_ml['n'])
print('ML — log-loss (CV):', {k: round(v, 3) for k, v in reporte_ml.get('cv', {}).items()})
for aviso in reporte_ml.get('avisos', []):
    print('  ⚠', aviso)


## 6. Pronóstico por partido pendiente

Para cada partido aún no jugado: **P(1/X/2)** del ensemble (Elo + Dixon-Coles +
ML), goles esperados y marcador más probable.

In [ ]:
from models import pronostico_partidos
import pandas as pd

tabla_partidos = pronostico_partidos(dataset, equipos, dc, modelos_ml)
print('Partidos pendientes de jugar:', len(tabla_partidos))
tabla_partidos.head(20)


## 7. Simulación Monte Carlo del torneo

Completa los partidos no jugados, resuelve los grupos con el **desempate oficial
FIFA**, asigna los **8 mejores terceros**, arma el cuadro de Eliminatorias y
simula hasta la final (prórroga/penales resueltos por fuerza). Los partidos ya
jugados quedan **fijos**.

> Subí `N_SIMS` a 50000 para mayor precisión (tarda ~20-30 s).

In [ ]:
from simulate import simular_torneo

N_SIMS = 20000
resultados = simular_torneo(equipos, datos.fixture, datos.bracket, dc,
                            n_sims=N_SIMS, semilla=2026)
print('Corridas:', resultados['n_sims'])


## 8. 🏆 Ranking de probabilidad de ser CAMPEÓN

In [ ]:
tabla_campeon = resultados['campeon'].copy()
tabla_campeon['prob_campeon_%'] = (tabla_campeon['prob_campeon'] * 100).round(2)
tabla_campeon[['pais', 'prob_campeon_%']].head(20)


## 9. Probabilidad de alcanzar cada ronda

In [ ]:
cols_ronda = ['pais', 'prob_32avos', 'prob_16avos', 'prob_Cuartos',
              'prob_Semifinales', 'prob_Final', 'prob_campeon']
av = resultados['avance'][cols_ronda].copy()
for c in cols_ronda[1:]:
    av[c] = (av[c] * 100).round(1)
av.head(16)


## 10. Gráficos (se guardan en `outputs/`)

In [ ]:
from viz import grafico_campeon, heatmap_avance

ruta1 = grafico_campeon(resultados['campeon'], top=15)
ruta2 = heatmap_avance(resultados['avance'], top=20)
print('Guardado en:', ruta1, 'y', ruta2)


## 11. Probabilidades por grupo (gana grupo / clasifica)

In [ ]:
g = resultados['grupos'].copy()
g['gana_grupo_%'] = (g['prob_gana_grupo'] * 100).round(1)
g['clasifica_%'] = (g['prob_clasifica'] * 100).round(1)
g[['grupo', 'pais', 'gana_grupo_%', 'clasifica_%']]


## 12. Guardar resultados a `outputs/`

In [ ]:
os.makedirs('outputs', exist_ok=True)
tabla_partidos.to_csv('outputs/pronostico_partidos.csv', index=False)
resultados['campeon'].to_csv('outputs/prob_campeon.csv', index=False)
resultados['avance'].to_csv('outputs/prob_avance.csv', index=False)
resultados['grupos'].to_csv('outputs/prob_grupos.csv', index=False)
print('CSVs de salida guardados en outputs/.')


---
## ♻️ Cómo actualizar el pronóstico

1. Abrí `Mundial_2026_fuente_datos.xlsx` y cargá los goles de los partidos
   jugados en la hoja **Fixture_Grupos** (columnas *Goles A* / *Goles B*) y/o en
   **Eliminatorias**.
2. Commiteá y pusheá el Excel al repo (la `raw URL` siempre toma el último commit).
3. En Colab: **Entorno de ejecución ▸ Ejecutar todo**. Las probabilidades se
   recalculan solas: los partidos con goles cargados quedan fijos y los vacíos se
   vuelven a simular.

*Entrega probabilidades, no recomendaciones de apuestas.*
